In [1]:
import os
import polars as pl
import datetime
import decimal
import uuid
from pyspark.sql import functions as F
from pyspark.sql.functions import col, date_trunc, when
from src.utils.config import setup_clickhouse_client
from src.utils.spark import get_spark

spark = get_spark()

ch_user = os.getenv('CLICKHOUSE_USER')
ch_password = os.getenv('CLICKHOUSE_PW')
ch_host = os.getenv('CLICKHOUSE_HOST')
client = setup_clickhouse_client(ch_user, ch_password, ch_host)

In [3]:
def convert_row(row):
    return [
        x.isoformat() if isinstance(x, datetime.datetime)
        else float(x) if isinstance(x, decimal.Decimal)
        else str(x) if isinstance(x, uuid.UUID) 
        else ','.join(map(str, x)) if isinstance(x, list)
        else x
        for x in row
    ]

In [10]:
# Fetch flights data
flights_query = "SELECT * FROM aviation_flights order by (dep_scheduled_time, dep_actual_time, arr_scheduled_time, arr_actual_time)"
flights_result = client.query(flights_query).result_rows
flights_columns = client.query(flights_query).column_names
flights_result = [convert_row(row) for row in flights_result]

# Create spark DataFrame
flights_pdf = pl.DataFrame(flights_result, schema=flights_columns, orient='row')

print(f"Total flight records retrieved: {len(flights_pdf)}")
flights_pdf.head(5)

Total flight records retrieved: 129442


id,insert_time,flight_type,status,iata_number,airline_name,dep_scheduled_time,dep_actual_time,dep_delay_mins,arr_scheduled_time,arr_actual_time,arr_delay_mins,codeshared_airline,codeshared_flight_number
str,str,str,str,str,str,str,str,i64,str,str,i64,str,str
"""6d6eb7ea-694b-4050-b2bf-2a5cb5…","""2026-04-10T14:49:47.510000+08:…","""arrival""","""landed""","""ua1""","""united airlines""","""2026-02-27T14:35:00""","""2026-02-27T14:57:00""",22,"""2026-03-01T00:01:00""","""2026-02-28T23:35:00""",0,"""""",""""""
"""3c71e857-3e4f-41cb-9fcf-0130d2…","""2026-04-10T14:50:16.484000+08:…","""arrival""","""landed""","""sq27""","""singapore airlines""","""2026-02-28T00:45:00""","""2026-02-28T01:18:00""",33,"""2026-03-01T09:45:00""","""2026-03-01T09:33:00""",0,"""""",""""""
"""2d6be94c-3555-4311-95aa-13c80e…","""2026-04-10T14:50:16.484000+08:…","""arrival""","""landed""","""sq21""","""singapore airlines""","""2026-02-28T01:35:00""","""2026-02-28T02:03:00""",29,"""2026-03-01T09:45:00""","""2026-03-01T09:25:00""",0,"""""",""""""
"""70b0b485-efc0-4753-8280-a4c9e5…","""2026-04-10T14:50:24.761000+08:…","""arrival""","""landed""","""sq31""","""singapore airlines""","""2026-02-28T01:40:00""","""2026-02-28T02:10:00""",31,"""2026-03-01T11:05:00""","""2026-03-01T10:44:00""",0,"""""",""""""
"""d70fe84e-694d-4020-97e6-6a879e…","""2026-04-10T14:50:28.880000+08:…","""arrival""","""landed""","""ua29""","""united airlines""","""2026-02-28T02:35:00""","""2026-02-28T04:13:00""",98,"""2026-03-01T12:15:00""","""2026-03-01T12:49:00""",34,"""""",""""""


In [11]:
weather_query = "SELECT * FROM historical_weather_data order by date_observed"

weather_result = client.query(weather_query).result_rows
weather_result = [convert_row(row) for row in weather_result]
weather_columns = client.query(weather_query).column_names

weather_pdf = pl.DataFrame(weather_result, schema=weather_columns, orient='row')

print(f"Total weather records retrieved: {len(weather_pdf)}")
weather_pdf.head(5)

Total weather records retrieved: 966


id,insert_time,date_observed,current_temp,feels_like_temp,pressure_hPa,humidity_pct,max_temp,min_temp,wind_speed_ms,wind_deg,cloudiness_pct,rain_1h,rain_3h,weather_main,weather_desc
str,str,str,f64,f64,i64,i64,f64,f64,f64,i64,i64,f64,f64,str,str
"""5cc74c3f-b81e-48c6-ac3f-0c2d65…","""2026-04-10T12:45:38.549000+08:…","""2026-02-28T16:00:00""",27.79,31.72,1009,81,26.92,27.97,3.09,30,75,0.0,0.0,"""Clouds""","""broken clouds"""
"""ebec1e01-2b88-42cb-8772-f39638…","""2026-04-10T12:45:38.549000+08:…","""2026-02-28T17:00:00""",27.22,30.79,1009,85,26.92,27.74,2.57,70,75,0.0,0.0,"""Clouds""","""broken clouds"""
"""eb52ac12-8740-498b-ae0d-aab525…","""2026-04-10T12:45:38.549000+08:…","""2026-02-28T18:00:00""",27.22,30.9,1008,86,26.92,27.74,2.57,10,75,0.0,0.0,"""Clouds""","""broken clouds"""
"""7a20d0bb-1920-4d87-91cc-0afaff…","""2026-04-10T12:45:38.549000+08:…","""2026-02-28T19:00:00""",27.22,30.9,1007,86,25.92,27.74,2.57,350,75,0.0,0.0,"""Clouds""","""broken clouds"""
"""b2b7e1f2-5e3d-445f-8c5c-a3363c…","""2026-04-10T12:45:38.549000+08:…","""2026-02-28T20:00:00""",26.93,30.34,1007,88,25.92,27.18,2.57,340,75,0.0,0.0,"""Clouds""","""broken clouds"""


In [12]:
print('Prepared flights and weather data for merging.')

# Filter out cancelled and unknown status flights
valid_statuses = ['cancelled', 'unknown']
flights_filtered = flights_pdf.filter(~pl.col('status').is_in(valid_statuses)) \
                    .unique(subset=['flight_type', 'status', 'iata_number', 'airline_name', 
                                    'dep_scheduled_time', 'arr_scheduled_time', 'dep_actual_time', 'arr_actual_time'],
                                    keep='first')

print(f"Flight records after filtering: {len(flights_filtered)}")

Prepared flights and weather data for merging.
Flight records after filtering: 126064


In [13]:
# Create hour columns for both departure and arrival times
flights_filtered = flights_filtered.with_columns(
    pl.col('dep_scheduled_time').str.to_datetime().dt.truncate('1h').alias('dep_hour'),
    pl.col('arr_scheduled_time').str.to_datetime().dt.truncate('1h').alias('arr_hour')
)

# Split into departure and arrival flights for conditional merging 
departure_flights = flights_filtered.filter(pl.col('flight_type') == 'departure')
arrival_flights = flights_filtered.filter(pl.col('flight_type') == 'arrival')

print(f"\nDeparture flights: {len(departure_flights)}, Arrival flights: {len(arrival_flights)}")
departure_flights.head(3)
arrival_flights.head(3)


Departure flights: 63390, Arrival flights: 62674


id,insert_time,flight_type,status,iata_number,airline_name,dep_scheduled_time,dep_actual_time,dep_delay_mins,arr_scheduled_time,arr_actual_time,arr_delay_mins,codeshared_airline,codeshared_flight_number,dep_hour,arr_hour
str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,datetime[μs],datetime[μs]
"""53687328-1109-497b-ba22-404e0d…","""2026-04-10T14:52:49.876000+08:…","""arrival""","""landed""","""tr245""","""scoot""","""2026-03-03T08:35:00""","""2026-03-03T08:33:00""",0,"""2026-03-03T11:00:00""","""2026-03-03T10:41:00""",0,"""""","""""",2026-03-03 08:00:00,2026-03-03 11:00:00
"""6a23a8c1-7b51-405b-ab37-249cfc…","""2026-04-10T14:59:45.393000+08:…","""arrival""","""landed""","""va5671""","""virgin australia""","""2026-03-08T08:55:00""","""2026-03-08T09:07:00""",12,"""2026-03-08T14:30:00""","""2026-03-08T14:02:00""",0,"""singapore airlines""","""sq601""",2026-03-08 08:00:00,2026-03-08 14:00:00
"""5e99238a-3d75-42ab-812e-510024…","""2026-04-10T14:59:17.786000+08:…","""arrival""","""landed""","""sq957""","""singapore airlines""","""2026-03-08T03:10:00""","""2026-03-08T03:18:00""",8,"""2026-03-08T06:00:00""","""2026-03-08T05:33:00""",0,"""""","""""",2026-03-08 03:00:00,2026-03-08 06:00:00


In [14]:
# need to trace why insert_time, date_observed, and weather_hour, weather_main, weather_desc are null after conversion to polars
weather_pdf = weather_pdf.with_columns(
    pl.col('date_observed').str.to_datetime().dt.truncate('1h').alias('weather_hour') 
)

# Deduplicate weather data: keep only one record per hour
# find out how to concatenate unique weather conditions and descriptions for duplicate hour/date entries
weather_pdf = weather_pdf.group_by(['weather_hour', 'date_observed']).agg(
    [
        pl.exclude(["id", "insert_time", "weather_main", "weather_desc", "weather_hour", "date_observed"]).mean(),

        pl.col("weather_main")
        .drop_nulls()
        .unique()
        .str.join(", ")
        .alias("weather_main"),

        pl.col("weather_desc")
        .drop_nulls()
        .unique()
        .str.join(", ")
        .alias("weather_desc"),
    ]
) 
print(f"Weather records by hour: {len(weather_pdf)}")

weather_pdf.head(3)

Weather records by hour: 960


weather_hour,date_observed,current_temp,feels_like_temp,pressure_hPa,humidity_pct,max_temp,min_temp,wind_speed_ms,wind_deg,cloudiness_pct,rain_1h,rain_3h,weather_main,weather_desc
datetime[μs],str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str
2026-03-07 05:00:00,"""2026-03-07T05:00:00""",31.22,36.12,1011.0,63.0,29.96,31.97,5.14,10.0,75.0,0.0,0.0,"""Clouds""","""broken clouds"""
2026-03-09 05:00:00,"""2026-03-09T05:00:00""",31.22,35.32,1012.0,60.0,29.96,31.97,6.17,360.0,75.0,0.0,0.0,"""Clouds""","""broken clouds"""
2026-03-20 11:00:00,"""2026-03-20T11:00:00""",30.36,34.67,1008.0,65.0,28.92,30.97,7.72,20.0,75.0,0.0,0.0,"""Clouds""","""broken clouds"""


In [ ]:
# Merge departure flights with weather
# Departure: join dep_hour with weather_hour (not exceeding scheduled time means weather_hour <= dep_hour)
if len(departure_flights) > 0:
    merged_departures = departure_flights.join(
        weather_pdf,
        how='left',
        left_on='dep_hour',
        right_on='weather_hour',
        maintain_order='left'
    )
    print(f"Merged departure flights: {len(merged_departures)}") # will need to check this tmr

# Merge arrival flights with weather
# Arrival: join arr_hour with weather_hour (same hour)
if len(arrival_flights) > 0:
    merged_arrivals = arrival_flights.join(
        weather_pdf,
        how='left',
        left_on='arr_hour',
        right_on='weather_hour',
        maintain_order='left'
    )
    print(f"Merged arrival flights: {len(merged_arrivals)}")

# Combine both datasets
if len(merged_departures) > 0 and len(merged_arrivals) > 0:
    combined_flights = pl.concat([merged_departures, merged_arrivals], how='vertical') \
            .sort(by=['dep_scheduled_time', 'dep_actual_time', 'arr_scheduled_time', 'arr_actual_time'], nulls_last=True)
    # print(combined_flights.columns)
    combined_flights = combined_flights.drop(['id', 'insert_time', 'iata_number', 'airline_name', 'codeshared_airline', 'codeshared_flight_number'])
else:
    combined_flights = pl.DataFrame()

print(f"\n Combined flights with weather data: {len(combined_flights)} rows")
combined_flights.head(3)

Merged departure flights: 63390
Merged arrival flights: 62674
['id', 'insert_time', 'flight_type', 'status', 'iata_number', 'airline_name', 'dep_scheduled_time', 'dep_actual_time', 'dep_delay_mins', 'arr_scheduled_time', 'arr_actual_time', 'arr_delay_mins', 'codeshared_airline', 'codeshared_flight_number', 'dep_hour', 'arr_hour', 'date_observed', 'current_temp', 'feels_like_temp', 'pressure_hPa', 'humidity_pct', 'max_temp', 'min_temp', 'wind_speed_ms', 'wind_deg', 'cloudiness_pct', 'rain_1h', 'rain_3h', 'weather_main', 'weather_desc']

 Combined flights with weather data: 126064 rows


flight_type,status,dep_scheduled_time,dep_actual_time,dep_delay_mins,arr_scheduled_time,arr_actual_time,arr_delay_mins,dep_hour,arr_hour,date_observed,current_temp,feels_like_temp,pressure_hPa,humidity_pct,max_temp,min_temp,wind_speed_ms,wind_deg,cloudiness_pct,rain_1h,rain_3h,weather_main,weather_desc
str,str,str,str,i64,str,str,i64,datetime[μs],datetime[μs],str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str
"""arrival""","""landed""","""2026-02-27T14:35:00""","""2026-02-27T14:57:00""",22,"""2026-03-01T00:01:00""","""2026-02-28T23:35:00""",0,2026-02-27 14:00:00,2026-03-01 00:00:00,"""2026-03-01T00:00:00""",26.36,26.36,1008.0,90.0,25.92,27.18,2.57,40.0,75.0,0.0,0.0,"""Clouds""","""broken clouds"""
"""arrival""","""landed""","""2026-02-28T00:45:00""","""2026-02-28T01:18:00""",33,"""2026-03-01T09:45:00""","""2026-03-01T09:33:00""",0,2026-02-28 00:00:00,2026-03-01 09:00:00,"""2026-03-01T09:00:00""",31.81,37.85,1005.0,64.0,30.92,31.97,4.12,110.0,75.0,0.0,0.0,"""Clouds""","""broken clouds"""
"""arrival""","""landed""","""2026-02-28T01:35:00""","""2026-02-28T02:03:00""",29,"""2026-03-01T09:45:00""","""2026-03-01T09:25:00""",0,2026-02-28 01:00:00,2026-03-01 09:00:00,"""2026-03-01T09:00:00""",31.81,37.85,1005.0,64.0,30.92,31.97,4.12,110.0,75.0,0.0,0.0,"""Clouds""","""broken clouds"""
